# ML-09 — Validation and Research Claim Audit

## 1. Two paper findings + my methodology questions

### Finding #1 — The Anatomy of Growing Content

The paper reports that growing content was longer and younger than declining content. The comparison used 74,187 rising pages and 45,272 falling pages. Growing pages averaged about 3.2K words and 184 days old, while declining pages averaged about 2.3K words and 230 days old.

**My methodology question:** How exactly is the growing/declining label constructed, and are any of the variables used to construct that label also being used in the comparison? I would verify that the grouping rule is independent from the characteristics being compared.

The large sample provides useful observed evidence, but this is an observational comparison. It supports a directional association rather than a causal claim that longer content or younger age causes growth.

### Finding #2 — The Content Performance Curve

The paper reports that content reaches its strongest health-score level around 61–90 days, declines after 270 days, and shows a rebound among 365+ day content. The paper also qualifies the rebound by noting that older pages can recover when they have been refreshed.

**My methodology question:** Do the age buckets represent the same pages followed over time, or different pages observed at different ages? I would also want to know whether refreshed and unrefreshed pages differ systematically.

An observational comparison across age buckets can show measured differences without proving that age itself causes the performance change. The narrower interpretation of the 365+ rebound is therefore important.

### Why these questions matter

These questions focus on whether the label or grouping is constructed independently and whether the validation design supports the strength of the conclusion. The goal is constructive: make the evidence easier to reproduce and interpret safely.

In [ ]:
print('Finding #1 and Finding #2 methodology review completed.')
print('Both findings are treated as observational evidence, not causal proof.')

## 2. My model under an honest split (before/after)

The Week-5 Random Forest was originally evaluated using a random train/test split. I then evaluated the same model and feature set using a client-grouped split so that no client appeared in both training and testing.

The random split measured MAE 1.400, RMSE 20.096, and R² 0.909. The client-grouped split measured MAE 1.134, RMSE 23.509, and R² 0.867, with 25 training clients, 7 test clients, and 0 overlapping clients.

The results are mixed rather than uniformly better or worse. MAE decreased, while RMSE increased and R² decreased. Therefore, the change in validation design changes the measured error profile and should not be described simply as an improvement.

For the grouped test set, the naive mean-prediction baseline measured MAE 21.592, RMSE 64.622, and R² −0.007. The Random Forest therefore showed substantially lower measured error than this naive baseline on the grouped test set.

These measurements are directional decision-support evidence for unseen clients, not proof of deployment-ready performance.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Validation': ['Week-5 Random Split', 'Client-Grouped Split', 'Grouped Mean Baseline'],
    'MAE': [1.3997466666666667, 1.1344710368327113, 21.591770221616635],
    'RMSE': [20.095635061292953, 23.509109005721616, 64.62204641802933],
    'R2': [0.9089571237542073, 0.866703338047662, -0.007182910658168673]
})

comparison

print('Training clients: 25')
print('Test clients: 7')
print('Overlapping clients: 0')

## 3. Leakage audit

The same leakage hunt from Week 3 is applied to the final Week-5 feature set.

**Target:** `clicks_90d`

**Model features:** `impressions_90d`, `sessions_90d`, `content_age_days`, `avg_position`, `ctr`, and `trend_pct`.

Known label-derived fields include `trend_direction`, `trend_pct`, and `is_declining_label`. `trend_direction` and `is_declining_label` were not used as model features.

The feature/target correlation check showed the strongest observed relationships for `sessions_90d` (0.766) and `impressions_90d` (0.696). Correlation alone does not establish leakage.

The main remaining concern is temporal leakage. Several performance variables are rolling or windowed measurements, and the starter dataset does not provide enough prediction-time information in this notebook to fully reconstruct whether every feature was available before the target window.

Therefore, I do not claim that the feature set is proven leakage-free. The audit found no direct use of the known derived label fields, but temporal alignment remains a limitation that should be resolved before deployment claims.

In [ ]:
target = 'clicks_90d'
features = [
    'impressions_90d',
    'sessions_90d',
    'content_age_days',
    'avg_position',
    'ctr',
    'trend_pct'
]

correlations = {
    'clicks_90d': 1.000000,
    'sessions_90d': 0.765661,
    'impressions_90d': 0.696281,
    'ctr': 0.010609,
    'trend_pct': -0.001778,
    'content_age_days': -0.022678,
    'avg_position': -0.099304
}

print('Target:', target)
print('\nModel features:')
for feature in features:
    print('-', feature)

print('\nKnown label-derived fields:')
print('- trend_direction')
print('- trend_pct')
print('- is_declining_label')

print('\nFeature/target correlation check:')
print(pd.Series(correlations).sort_values(ascending=False))

## 4. Claim rewrite

My original Week-5 result could be read as saying that the Random Forest substantially outperformed the baseline and achieved strong predictive performance.

A more careful claim is:

> On the available starter dataset, the Random Forest showed lower measured error than the simple mean-prediction baseline on the tested validation designs. On the client-grouped split, it achieved an MAE of 1.134 and RMSE of 23.509, compared with 21.592 and 64.622 for the grouped mean-prediction baseline.

I would not claim that the model is deployment-ready or that it will generalize equally well to future clients. The grouped validation provides directional decision-support evidence for unseen clients, but it does not establish a fully time-separated prediction setup.

I also would not claim that the model has no leakage. The audit found no direct use of the known derived label fields, but temporal leakage remains a limitation because the timing of several rolling performance features cannot be fully reconstructed from the starter dataset.

Therefore, the strongest supported conclusion is that the Random Forest showed measured predictive value relative to a naive baseline under the tested validation designs, while stronger temporal validation is needed before broader claims are made.

In [ ]:
print('Claim rewrite uses observed, measured, directional, and decision-support language.')
print('Deployment readiness is not claimed.')

## Self-check

Before submission, confirm each line honestly:

- [x] Two research-paper findings are named with constructive methodology questions.
- [x] The Week-5 model is compared under random and client-grouped validation.
- [x] Before/after validation results are shown with measured metrics.
- [x] The grouped split has 25 training clients, 7 test clients, and 0 overlapping clients.
- [x] A naive mean-prediction baseline is reported for the grouped test set.
- [x] The final feature set is audited for label-derived and temporal leakage.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] No private client names or private queries are included.
- [ ] The notebook runs top to bottom with no errors.
- [ ] The completed notebook is committed under `work/notebooks/w06_validation_audit.ipynb`.
- [ ] Changes are pushed to the repository.